# MYSTRAN and NASTRAN airplane demo 

this is a copy of the demo model provided with sdynpy

```python
import sdynpy.demo as demo
system = demo.beam_airplane.system
geometry = demo.beam_airplane.geometry
```

works with base5 conda env
development environment notes

conda install -c conda-forge jupyterlab
conda activate base5

```python
import sys, pathlib, importlib
repo_root = pathlib.Path('../../..').resolve()
libdir = str(repo_root / 'src')
sys.path.insert(0, libdir)
import sdynpy as sdpy
importlib.reload(sdpy)
```

or even better, do this so it only imports the dev lib and not cconfuses it with others  

```python
(base5) python -m pip uninstall sdynpy
(base5) python -m pip install -e C:\Users\user\github\sdynpy
```  

# Imports

In [ ]:
import sdynpy as sdpy

In [ ]:
print(sdpy.__file__)   # should be ...\github\sdynpy\src\sdynpy\__init__.py

In [ ]:
pwd

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import importlib
import sys, pathlib

In [ ]:
repo_root = pathlib.Path('../../..').resolve()
repo_root

In [ ]:
libdir = str(repo_root / 'src')
libdir

In [ ]:
# import local dev library

sys.path.insert(0, libdir)
import sdynpy as sdpy
importlib.reload(sdpy)

In [ ]:
%load_ext autoreload
%autoreload 2
import sdynpy as sdpy

In [ ]:
print(sdpy.__file__)   # should be ...\github\sdynpy\src\sdynpy\__init__.py

In [ ]:
pwd

# load sdynpy demo airplane

In [ ]:
import sdynpy.demo as demo
geometry0 = demo.beam_airplane.geometry
system0 = demo.beam_airplane.system

# plotting params
plot_kwargs={'node_size':1, 'view_from':[-1,1,1]}

# Get the airplane system and geometry from the demo subpackage
geometry0 = demo.beam_airplane.geometry


In [ ]:
# Create a modal model to make integration faster
shapes0 = system0.eigensolution(maximum_frequency = 10)
shapes0.damping = 0.02
modal_system0 = shapes0[6:].system()
shapes0[6:]

In [ ]:
geometry0.node[:10]

In [ ]:
geometry0.plot(label_nodes=True)

## export demo model to csv to read externally

In [ ]:

n = geometry0.node
df = pd.DataFrame({
    'id': n.id,
    'x': n.coordinate[:, 0],
    'y': n.coordinate[:, 1],
    'z': n.coordinate[:, 2],
    'def_cs': n.def_cs,   # CP: frame the position is defined in
    'disp_cs': n.disp_cs, # CD: frame the DOFs/displacements are defined in
    'color': n.color,
}).set_index('id')
df.to_csv('demo_airplane_grids_export.csv')

# Read Mystran dat and f06 demo airplane model

In [ ]:
mystrandatfilename = repo_root.joinpath('docs/book/example_models/sdynpy_airplane_modes/sdynpy_airplane_modes_mystran.dat')
print(mystrandatfilename)
geometry_mys, shapes_mys = sdpy.mystran.read_mystran(mystrandatfilename, source='f06', damping=0.005)
# print(shapes.frequency)
shapes_mys

In [ ]:
shapes_mys = shapes_mys[6:]
shapes_mys

## save sdynpy geometry for use later

In [ ]:
geometry_mys.save('demo_airplane_geometry.npz')

## plot mystran model with grids

In [ ]:
geometry_mys.plot(label_nodes=True, node_size=8, label_font_size=10)

In [ ]:
# Every translation DOF at every node (default)
geometry_mys.plot_coordinate(label_dofs=True, label_font_size=10, arrow_scale=0.02)

In [ ]:
# only plot Z+
coords = sdpy.coordinate_array(geometry_mys.node.id, 'Z+')
geometry_mys.plot_coordinate(coords, label_dofs=True, arrow_scale=0.02)

In [ ]:
# Specific DOFs, e.g. all three translations and rotations at selected nodes
excitation_dofs = sdpy.coordinate_array([241, 201], ['Z+', 'Z+', ])
excitation_dofs = sdpy.coordinate_array( string_array = ['241Z+','201Z+'])
excitation_dofs = sdpy.coordinate_array([241, 201], 'Z+')
coords
geometry_mys.plot_coordinate(excitation_dofs, label_dofs=True, arrow_scale=0.03, label_font_size=10, 
                         plot_kwargs=plot_kwargs)

In [ ]:
geometry_mys.node.disp_cs

In [ ]:
# Only nodes that use a specific displacement CS
nodes = geometry_mys.node.id[geometry_mys.node.disp_cs == 5]
geometry_mys.plot_coordinate(sdpy.coordinate.from_nodelist(nodes), label_dofs=True)

In [ ]:
geometry_mys.plot_coordinate(label_dofs=True)
# geometry_mys.plot_coordinate(load_locations,,plot_kwargs=plot_kwargs)
# Draw the response to the excitation on the structure

In [ ]:
# response_dofs = sdpy.coordinate.from_nodelist(geometry_mys.node.id)
response_dofs = sdpy.coordinate_array( string_array = ['275Z+','249Z+','101Z+','101Y+'])
response_dofs

In [ ]:
geometry_mys.plot_coordinate(response_dofs,label_dofs=True, arrow_scale=0.03, label_font_size=10, 
                         plot_kwargs=plot_kwargs)

## animate mystran f06 modes

In [ ]:
p = geometry_mys.plot_shape(shapes, plot_kwargs=plot_kwargs, 
                        background_plotter_kwargs={'editor': False, 'window_size': (800, 600)})
# Add your own text (any corner or position)
p.add_text('sdynpy demo airplane mode shapes', position='upper_right', font_size=14, color='k', name='title')
# Or place it in pixel/viewport coordinates
p.add_text('MYSTRAN solver', position='lower_right', font_size=12, name='tag')  # position=(10, 10)
# p.app_window.resize(1600, 900)
p.add_axes()                          # small XYZ triad in the corner


#### optionally export gif of animation
# p.save_animation('mode1.gif', frames=30, frame_rate=20)            # current shape
# p.save_animation_all_shapes('Shape_{:}.gif', frames=30, frame_rate=20)  # every shape


### running unattended
# p = geometry_mys.plot_shape(shapes, background_plotter_kwargs={'editor': False, 'show': False})
# p.save_animation_all_shapes('Shape_{:}.gif')
# p.close()

## read mystran OU4 file with modal results

In [ ]:
# The OU4 results should match the F06 results
_, shapes_ou4 = sdpy.mystran.read_mystran(mystrandatfilename, source='ou4', damping=0.005)
print(sdpy.shape.mac(shapes, shapes_ou4).diagonal())

In [ ]:
shapes_ou4 = shapes_ou4[6:]
shapes_ou4

# read nastran model

In [ ]:
nastrandatfilename = repo_root.joinpath('docs/book/example_models/sdynpy_airplane_modes/sdynpy_airplane_modes_nastran.dat')
print(nastrandatfilename)
geometry_nas, shapes_nas = sdpy.nastran.read_nastran(nastrandatfilename, source='f06', damping=0.005)
# print(shapes.frequency)
shapes_nas

In [ ]:
geometry_nas, shapes_nas = sdpy.nastran.read_nastran(nastrandatfilename, source='pch', damping=0.005)
shapes_nas

In [ ]:
geometry.plot_shape(shapes_nas)

In [ ]:
# Compare the two solvers
mac = sdpy.shape.mac(shapes_nas, shapes_m)


# mystran compute FRF

In [ ]:
# Acceleration FRFs between any pair of degrees of freedom from the unconstrained DOFs.

frequencies = np.arange(0, 10, 0.1)
responses = sdpy.coordinate_array([201, 241], 'Z+')
references = sdpy.coordinate_array(201, 'Y+')
frfs = shapes.compute_frf(frequencies, responses, references)
frfs.plot();
